# AMD ROCm 环境自检 Notebook

用途：在 Radeon Cloud / AMD GPU Jupyter 环境里确认当前实例能不能跑 ROCm 训练。

先不要下载 WeClone、不要训练模型。把下面单元格从上到下跑完，重点看：

- GPU 型号和显存
- `torch.version.hip` 是否有值
- `torch.cuda.is_available()` 是否为 `True`
- 矩阵乘法是否真的跑在 GPU 上
- 当前显存大概适合 7B LoRA、7B QLoRA，还是只能先做小模型测试

说明：PyTorch ROCm 版本仍然沿用 `torch.cuda.*` API 名字，所以 AMD 卡上看到 `cuda` 字样是正常的。关键看 `torch.version.hip`。

In [ ]:
import os
import sys
import json
import shutil
import platform
import subprocess
from textwrap import indent

def run(cmd, timeout=30):
    print(f"$ {cmd}")
    try:
        result = subprocess.run(
            cmd,
            shell=True,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=timeout,
        )
        print(result.stdout.strip() or "<no output>")
        print(f"exit_code={result.returncode}")
    except Exception as exc:
        print(type(exc).__name__, exc)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)
print("CWD:", os.getcwd())

## 1. 系统层查看 GPU

`rocm-smi` 最关键。如果不存在，通常说明当前镜像没有完整 ROCm 工具链，或者容器没有挂 GPU。

In [ ]:
run("which rocm-smi || true")
run("rocm-smi || true", timeout=60)
run("rocm-smi --showproductname --showmeminfo vram --showuse || true", timeout=60)
run("which amd-smi || true")
run("amd-smi static || true", timeout=60)
run("lspci | grep -Ei 'vga|3d|display|amd|ati' || true")

## 2. ROCm / HIP / PyTorch 自检

判断标准：

- `torch.version.hip` 有版本号：这是 ROCm 版 PyTorch。
- `torch.cuda.is_available()` 为 `True`：PyTorch 能看到 GPU。
- `torch.cuda.get_device_name(0)` 显示 AMD GPU 名称。

In [ ]:
try:
    import torch
    print("torch.__version__:", torch.__version__)
    print("torch.version.cuda:", torch.version.cuda)
    print("torch.version.hip:", getattr(torch.version, "hip", None))
    print("torch.cuda.is_available():", torch.cuda.is_available())
    print("torch.cuda.device_count():", torch.cuda.device_count())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"device {i}:", torch.cuda.get_device_name(i))
            props = torch.cuda.get_device_properties(i)
            print("  total_memory_gb:", round(props.total_memory / 1024**3, 2))
            print("  multi_processor_count:", props.multi_processor_count)
except Exception as exc:
    print(type(exc).__name__, exc)

## 3. GPU 矩阵计算冒烟测试

如果这里失败，就先别跑 WeClone。训练会更容易失败。

In [ ]:
import time
import torch

assert torch.cuda.is_available(), "PyTorch 没有识别到 GPU"

device = torch.device("cuda:0")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("device:", device)
print("dtype:", dtype)
print("bf16 supported:", torch.cuda.is_bf16_supported())

size = 4096
a = torch.randn((size, size), device=device, dtype=dtype)
b = torch.randn((size, size), device=device, dtype=dtype)

torch.cuda.synchronize()
t0 = time.time()
c = a @ b
torch.cuda.synchronize()
dt = time.time() - t0

print("matmul shape:", tuple(c.shape))
print("elapsed_sec:", round(dt, 4))
print("allocated_gb:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("reserved_gb:", round(torch.cuda.memory_reserved() / 1024**3, 3))

del a, b, c
torch.cuda.empty_cache()

## 4. 估算能跑什么微调

这个判断是保守经验值，不是严格上限。第一版 WeClone 建议先跑普通 LoRA，不要先碰 4bit QLoRA。

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024**3
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(vram_gb, 2))

    if vram_gb >= 44:
        suggestion = "可以优先尝试 7B/8B LoRA，之后再评估 14B LoRA 或更长上下文。"
    elif vram_gb >= 22:
        suggestion = "适合先跑 7B/8B 普通 LoRA：batch_size=1，gradient_accumulation_steps=8~16，cutoff_len=512~1024。"
    elif vram_gb >= 14:
        suggestion = "可以尝试 7B LoRA 的低配参数；如果 OOM，把 cutoff_len 降到 512，并开启 gradient_checkpointing。"
    else:
        suggestion = "建议先用 1.5B/3B 小模型验证流程，7B 训练可能频繁 OOM。"

    print("建议:", suggestion)

## 5. 包版本检查

如果后面跑 WeClone，重点关注 `torch`、`transformers`、`peft`、`trl`、`accelerate`、`llamafactory`。没有装也没关系，这个 notebook 只是自检。

In [ ]:
packages = [
    "torch",
    "torchvision",
    "transformers",
    "datasets",
    "accelerate",
    "peft",
    "trl",
    "modelscope",
    "llamafactory",
    "weclone",
]

for pkg in packages:
    run(f"python -m pip show {pkg} | sed -n '1,8p' || true")

## 6. 把结果发给我

请把下面几项的输出贴回来：

- `rocm-smi --showproductname --showmeminfo vram --showuse`
- `torch.__version__`
- `torch.version.hip`
- `torch.cuda.get_device_name(0)`
- `total_memory_gb`
- 矩阵计算是否成功

有了这些信息，我就可以给你改一个真正跑 WeClone / LLaMA Factory 的 ROCm 训练 notebook。